In [2]:
import pandas as pd
import pickle
import re
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import FunctionTransformer

# --- Load and prepare data ---
data = pd.read_csv("C:/Users/yussy/fyp/sentiment.csv")
data['clean_comment'] = data['clean_comment'].fillna('')

# --- Split into train and test ---
X = data['clean_comment']
y = data['Sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- Define cleaning function ---
def basic_text_cleaning(text_series):
    return text_series.apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x.lower()).strip())

# --- Define and Create the Pipeline ---
sentiment_pipeline = Pipeline([
    ('cleaner', FunctionTransformer(basic_text_cleaning, validate=False)),
    ('tfidf', TfidfVectorizer()),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        solver='saga',
        max_iter=1000,
        random_state=42,
        C=1.9608286241914519,
        penalty='l1'
    ))
])

# --- Train the entire pipeline ---
print("Training the pipeline...")
sentiment_pipeline.fit(X_train, y_train)

# --- Training Set Results ---
print("\n--- Training Evaluation Results ---")
y_pred_train = sentiment_pipeline.predict(X_train)
print(classification_report(y_train, y_pred_train))

# --- Save the pipeline to .pkl ---
model_filename = 'sentiment_analyzer_pipeline.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(sentiment_pipeline, file)

print(f"\nModel saved successfully as '{model_filename}'")


Training the pipeline...

--- Training Evaluation Results ---
              precision    recall  f1-score   support

    negative       0.88      0.92      0.90      1003
     neutral       0.94      0.98      0.96      2371
    positive       0.96      0.89      0.92      1866

    accuracy                           0.93      5240
   macro avg       0.93      0.93      0.93      5240
weighted avg       0.94      0.93      0.93      5240


Model saved successfully as 'sentiment_analyzer_pipeline.pkl'
